# Train Jarvis's local voice model (one-time, runs on Colab's free GPU)

This is the **only** step in the whole local-voice-clone setup that touches an outside
connection. Run every cell below, in order, then download the two files it produces
(`jarvis.pth` and `jarvis.index`) and drop them into `voice_clone/rvc_models/` on your Mac.
After that, everything runs 100% locally.

Before starting: **Runtime -> Change runtime type -> T4 GPU** (free tier is fine).

You will also need a free ngrok auth token (https://dashboard.ngrok.com/get-started/your-authtoken)
to expose the training UI from Colab to your browser — paste it in the cell marked below.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

## 2. Clone RVC and install dependencies (Colab's T4 is an older-generation NVIDIA GPU -> cu118 track)

In [ ]:
%cd /content
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git
%cd /content/Retrieval-based-Voice-Conversion-WebUI
!python -m pip install torch==2.7.1+cu118 torchaudio==2.7.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!python -m pip install -r requirments_cu118_py312.txt
!python -m pip install pyngrok

## 3. Download pretrained base checkpoints (hubert + rmvpe — required for training)

In [ ]:
!python -m pip install --upgrade huggingface_hub
%cd /content/Retrieval-based-Voice-Conversion-WebUI
!hf download lj1995/VoiceConversionWebUI --revision main --include "hubert_base/*" --local-dir assets
!hf download lj1995/VoiceConversionWebUI rmvpe.pt --revision main --local-dir assets/rmvpe

## 4. Upload your dataset
Upload `jarvis_voice_reference.zip` produced by `scripts/generate_voice_reference_dataset.py` on your Mac.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = next(iter(uploaded))
dataset_dir = "/content/Retrieval-based-Voice-Conversion-WebUI/dataset/jarvis_voice_reference"
os.makedirs(dataset_dir, exist_ok=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(dataset_dir)
print("Dataset extracted to:", dataset_dir)

## 5. Launch the training UI and expose it via ngrok
**Paste your free ngrok auth token below before running this cell.**

In [ ]:
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

import subprocess, time
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

%cd /content/Retrieval-based-Voice-Conversion-WebUI
proc = subprocess.Popen(["python", "webui.py", "--noautoopen"])
time.sleep(15)  # give the Gradio server a moment to bind the port

public_url = ngrok.connect(7865)
print("Open this URL in your browser:", public_url)

## 6. In the UI that just opened
1. Go to the **Train** tab.
2. Set the experiment name to `jarvis`.
3. Set the training folder path to: `/content/Retrieval-based-Voice-Conversion-WebUI/dataset/jarvis_voice_reference`
4. Run **Process data** -> **Feature extraction** (this uses the hubert/rmvpe checkpoints from step 3) -> **Train model**.
5. Reasonable defaults for a ~15 minute dataset on a free T4: total epochs 200, save frequency 50, batch size left at the UI's suggested default.
6. Training takes roughly 20-40 minutes on a free T4. Leave this tab open until it finishes.

## 7. Download the trained model
Run this once training in step 6 has finished. It searches for the newest `.pth` weight file
and the newest `.index` feature file RVC produced (rather than a hardcoded path, since exact
output locations can vary by RVC version) and downloads both.

In [ ]:
import subprocess
from google.colab import files

pth = subprocess.run(
    ["bash", "-c", "find /content/Retrieval-based-Voice-Conversion-WebUI -name '*.pth' -newer /content/Retrieval-based-Voice-Conversion-WebUI/requirments_cu118_py312.txt | sort | tail -1"],
    capture_output=True, text=True,
).stdout.strip()
index = subprocess.run(
    ["bash", "-c", "find /content/Retrieval-based-Voice-Conversion-WebUI -name '*.index' -newer /content/Retrieval-based-Voice-Conversion-WebUI/requirments_cu118_py312.txt | sort | tail -1"],
    capture_output=True, text=True,
).stdout.strip()

print("Found weight file:", pth or "NONE — check step 6 completed successfully")
print("Found index file:", index or "NONE — check step 6 completed successfully")

if pth:
    files.download(pth)
if index:
    files.download(index)

print("\nRename the downloaded files to jarvis.pth / jarvis.index and place them in")
print("voice_clone/rvc_models/ on your Mac.")